In [ ]:
# LAB 07 

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



# LOAD DATA

X = np.load("X_features.npy")
y = np.load("y_labels.npy")

# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


# METRICS FUNCTION

def evaluate_model(model, X_train, X_test, y_train, y_test):

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results = {
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Train Precision": precision_score(y_train, train_pred, average="macro"),
        "Test Precision": precision_score(y_test, test_pred, average="macro"),
        "Train Recall": recall_score(y_train, train_pred, average="macro"),
        "Test Recall": recall_score(y_test, test_pred, average="macro"),
        "Train F1": f1_score(y_train, train_pred, average="macro"),
        "Test F1": f1_score(y_test, test_pred, average="macro"),
    }

    return results


# RANDOMIZED SEARCH (A2)

def tune_svm(X_train, y_train):

    param_dist = {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"]
    }

    model = SVC()

    random_search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        n_iter=5,
        cv=5,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    return random_search.best_estimator_


# TRAIN MODELS (A3)

models = {

    "SVM (Tuned)": tune_svm(X_train, y_train),

    "Decision Tree": DecisionTreeClassifier(max_depth=5),

    "Random Forest": RandomForestClassifier(n_estimators=100),

    "Naive Bayes": GaussianNB(),

    "MLP": MLPClassifier(max_iter=500),

    "AdaBoost": AdaBoostClassifier(n_estimators=100)
}


# EVALUATE ALL MODELS

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    results[name] = evaluate_model(model, X_train, X_test, y_train, y_test)


# DISPLAY RESULTS (TABLE)

df_results = pd.DataFrame(results).T

print("\nModel Comparison:\n")
print(df_results)


Model Comparison:

               Train Accuracy  Test Accuracy  Train Precision  Test Precision  \
SVM (Tuned)          0.942529       0.578947         0.948889        0.569841   
Decision Tree        0.931034       0.394737         0.933421        0.392857   
Random Forest        1.000000       0.500000         1.000000        0.441270   
Naive Bayes          0.643678       0.421053         0.649177        0.391190   
MLP                  1.000000       0.657895         1.000000        0.636111   
AdaBoost             0.896552       0.526316         0.902148        0.514286   

               Train Recall  Test Recall  Train F1   Test F1  
SVM (Tuned)        0.944683     0.625556  0.945177  0.563889  
Decision Tree      0.934156     0.414444  0.931683  0.394568  
Random Forest      1.000000     0.517778  1.000000  0.468582  
Naive Bayes        0.628042     0.453889  0.628906  0.414420  
MLP                1.000000     0.707778  1.000000  0.647179  
AdaBoost           0.902601     0.